# Task 5 – Loan Default Prediction

This notebook follows the same **Task 5 structure** as the uploaded reference notebook, adapted to the uploaded Loan Default dataset.

### Models covered
- Logistic Regression
- Decision Tree
- Bootstrap validation
- Random Forest
- AdaBoost
- KNN
- Model Comparison

### Classification evaluation
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- Classification Report

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

### Load Dataset

In [2]:

df = pd.read_csv("Loan_default.csv")

df

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


In [3]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Default"].value_counts())

Dataset Shape: (255347, 18)

Columns:
['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']

Missing Values:
LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64

Target Distribution:
Default
0    225694
1     29653
Name: count, dtype: int64


### Data Preparation

`Default` is the target column:
- `0` = No Default
- `1` = Default

`LoanID` is an identifier, so it is removed from the model.

The categorical columns are converted using One-Hot Encoding.

In [4]:
X = df.drop(columns=["LoanID", "Default"])
y = df["Default"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (204277, 16)
Testing Data: (51070, 16)


### Logistic Regression

In [5]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train, y_train)

print("Train Score:", logistic_model.score(X_train, y_train))
print("Test Score:", logistic_model.score(X_test, y_test))

Train Score: 0.885102091767551
Test Score: 0.885275112590562


In [6]:
y_pred_lr = logistic_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_lr, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_lr, zero_division=0))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Accuracy: 0.885275112590562
Precision: 0.608433734939759
Recall: 0.03405833754847412
F1-score: 0.06450582787801373

Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45139
           1       0.61      0.03      0.06      5931

    accuracy                           0.89     51070
   macro avg       0.75      0.52      0.50     51070
weighted avg       0.85      0.89      0.84     51070


Confusion Matrix:
[[45009   130]
 [ 5729   202]]


### Decision Tree

Decision Tree Classifier predicts whether a borrower will default by splitting the data using feature-based decision rules.

In [7]:
decision_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

decision_model.fit(X_train, y_train)

print("Train Score:", decision_model.score(X_train, y_train))
print("Test Score:", decision_model.score(X_test, y_test))

Train Score: 1.0
Test Score: 0.8016056393185823


In [8]:
decision_model_depth5 = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
])

decision_model_depth5.fit(X_train, y_train)

print("Train Score:", decision_model_depth5.score(X_train, y_train))
print("Test Score:", decision_model_depth5.score(X_test, y_test))

Train Score: 0.8848524307680258
Test Score: 0.8848834932445663


### BOOTSTRAP

#### Checking how reliable our model result is

In [9]:
# Create a bootstrap sample from the training data
X_sample, y_sample = resample(
    X_train,
    y_train,
    replace=True,
    random_state=42
)

bootstrap_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
])

bootstrap_model.fit(X_sample, y_sample)

bootstrap_score = bootstrap_model.score(X_test, y_test)

print("Bootstrap Score:", bootstrap_score)

Bootstrap Score: 0.8849226551791658


### Advanced Models

### Random Forest Algorithm

In [10]:
model_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        n_jobs=-1
    ))
])

model_rf.fit(X_train, y_train)

print("Train Score:", model_rf.score(X_train, y_train))
print("Test Score:", model_rf.score(X_test, y_test))

Train Score: 0.8838733680247899
Test Score: 0.8838652829449775


In [11]:
y_pred_rf = model_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_rf, zero_division=0))

              precision    recall  f1-score   support

           0       0.88      1.00      0.94     45139
           1       0.00      0.00      0.00      5931

    accuracy                           0.88     51070
   macro avg       0.44      0.50      0.47     51070
weighted avg       0.78      0.88      0.83     51070

Confusion Matrix:
[[45139     0]
 [ 5931     0]]
Accuracy: 0.8838652829449775
Precision: 0.0
Recall: 0.0
F1-score: 0.0


### Try Different Values of n_estimators

In [12]:
for n in [50, 100, 150, 200]:
    rf_test = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=n,
            max_depth=5,
            random_state=42,
            n_jobs=-1
        ))
    ])

    rf_test.fit(X_train, y_train)
    test_score = rf_test.score(X_test, y_test)

    print("Trees:", n, "Test Score:", test_score)

Trees: 50 Test Score: 0.8838652829449775
Trees: 100 Test Score: 0.8838652829449775
Trees: 150 Test Score: 0.8838652829449775
Trees: 200 Test Score: 0.8838652829449775


### AdaBoost

In [13]:
ada = Pipeline([
    ("preprocessor", preprocessor),
    ("model", AdaBoostClassifier(
        n_estimators=50,
        learning_rate=1.0,
        random_state=42
    ))
])

ada.fit(X_train, y_train)

print("Train Score:", ada.score(X_train, y_train))
print("Test Score:", ada.score(X_test, y_test))

Train Score: 0.8853027996299143
Test Score: 0.885549246132759


In [ ]:
y_pred_ada = ada.predict(X_test)

print("AdaBoost Accuracy:", accuracy_score(y_test, y_pred_ada))
print("AdaBoost Precision:", precision_score(y_test, y_pred_ada, zero_division=0))
print("AdaBoost Recall:", recall_score(y_test, y_pred_ada, zero_division=0))
print("AdaBoost F1-score:", f1_score(y_test, y_pred_ada, zero_division=0))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ada, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ada))

### KNN

In [ ]:
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
])

knn_model.fit(X_train, y_train)

print("Train Score:", knn_model.score(X_train, y_train))
print("Test Score:", knn_model.score(X_test, y_test))

y_pred_knn = knn_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

print("\nAccuracy:", accuracy_score(y_test, y_pred_knn))
print("Precision:", precision_score(y_test, y_pred_knn, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_knn, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred_knn, zero_division=0))

### Model Comparison

In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_model_depth5,
    "Random Forest": model_rf,
    "AdaBoost": ada,
    "KNN": knn_model
}

comparison = []

for name, model in models.items():
    pred = model.predict(X_test)

    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison)
comparison_df.sort_values(by="Accuracy", ascending=False)

### Final Conclusion

Compare the models using **Accuracy, Precision, Recall and F1-score**.

For a loan-default problem, do not select a model using Accuracy alone. **Recall for the Default = 1 class and F1-score** are also important because the dataset contains many more non-default records than default records.